# Kaggle runner: VLM pushback study (SC4001)

What this runs, in order: get the code, install pinned packages, check the GPU, download POPE and the COCO images it uses,
run the CPU tests, run the **pilot** (100 dev questions per model), then (once you set `DEV_MODEL`) the **dev run** in a resumable loop.
**Dev split only.** Nothing here can read the test split.

**Before you start (Kaggle notebook sidebar):**
1. *Settings → Accelerator*: **GPU T4** (x1 or x2; only one is used). *Settings → Internet*: **on** (needs a phone-verified account).
2. *Add-ons → Secrets*:
   - `GITHUB_TOKEN`: only if the code repo is private (a fine-grained token with read access to that one repo).
   - `RCLONE_CONF`: optional, the text of your rclone config, to copy results to Google Drive (see README). Kaggle cannot mount Drive.
3. Edit the settings cell below.

Every result row is written to disk after each batch. If the session dies, run all cells again: finished rows are skipped.

In [ ]:
# ---------------- Settings: edit these ----------------
REPO_URL = "https://github.com/<owner>/<repo>.git"  # the project repo
BRANCH = "main"
CODE_DATASET = None   # instead of GitHub: e.g. "/kaggle/input/vlmsyc-code" (a Kaggle dataset holding the repo)

PILOT_MODELS = ["qwen2vl_2b", "llava15_7b_4bit"]  # add "smolvlm" to try a third model
DEV_MODEL = None      # set after the pilot, e.g. "qwen2vl_2b"; None skips the dev run
SHARD = "0/1"         # teammates split the dev run: "0/3", "1/3", "2/3" (one each)

DRIVE_DIR = "gdrive:sc4001-vlm/results"  # rclone remote:folder for results; None = no Drive copy

In [ ]:
# ---------------- Platform and paths ----------------
import os, sys, subprocess, shutil, json, time

ON_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
# Kaggle also has the google.colab package installed, so check for a real Colab
# machine (this file only exists there) instead of trying the import.
ON_COLAB = not ON_KAGGLE and os.path.exists("/var/colab/hostname")
WORK = "/kaggle/working" if ON_KAGGLE else "/content" if ON_COLAB else os.getcwd()
REPO = f"{WORK}/vlmsyc"

if ON_COLAB:
    # On Colab, write results straight to Google Drive.
    from google.colab import drive
    drive.mount("/content/drive")
    RESULTS_DIR = "/content/drive/MyDrive/sc4001-vlm/results"
else:
    RESULTS_DIR = f"{WORK}/results"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.environ["RESULTS_DIR"] = RESULTS_DIR
os.environ["DATA_DIR"] = f"{REPO}/data"  # the repo's data/ folder (holds the committed splits.json)
print("platform:", "kaggle" if ON_KAGGLE else "colab" if ON_COLAB else "other")
print("RESULTS_DIR =", RESULTS_DIR)


def get_secret(name):
    """Read a secret from Kaggle or Colab; None if it is not set."""
    try:
        if ON_KAGGLE:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(name)
        if ON_COLAB:
            from google.colab import userdata
            return userdata.get(name)
    except Exception:
        return None

In [ ]:
# ---------------- Get the code ----------------
def git(*args, token=None):
    """Run git; hide the token if an error message contains it."""
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode != 0:
        msg = r.stderr.replace(token, "***") if token else r.stderr
        raise RuntimeError(f"git {args[0]} failed:\n{msg}")
    return r.stdout

if CODE_DATASET:
    shutil.copytree(CODE_DATASET, REPO, dirs_exist_ok=True)
else:
    token = get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
    if os.path.isdir(f"{REPO}/.git"):
        git("-C", REPO, "pull", "--ff-only", url, BRANCH, token=token)
    else:
        git("clone", "--branch", BRANCH, url, REPO, token=token)
        git("-C", REPO, "remote", "set-url", "origin", REPO_URL)  # do not keep the token on disk

os.chdir(REPO)
sys.path.insert(0, f"{REPO}/src")
print("code commit:", subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip() or "unknown")

In [ ]:
# ---------------- Install pinned packages ----------------
%pip install -q transformers==4.57.6 accelerate==1.10.1 bitsandbytes==0.49.2 pyyaml pytest

In [ ]:
# ---------------- Versions and GPU check ----------------
import importlib.metadata as md
import torch

for pkg in ["torch", "transformers", "accelerate", "bitsandbytes", "numpy", "pillow", "pyyaml"]:
    try:
        print(f"{pkg:>13}: {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:>13}: MISSING")
print("python:", sys.version.split()[0], "| CUDA build:", torch.version.cuda)
assert torch.cuda.is_available(), "No GPU: set Settings -> Accelerator -> GPU T4"
print("GPU:", torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
x = torch.ones(8, 8, dtype=torch.float16, device="cuda")
assert (x @ x).sum().item() == 512.0, "fp16 matmul gave a wrong answer"
print("fp16 matmul OK")
assert md.version("transformers") == "4.57.6", "transformers pin did not apply: restart the session and run again"

In [ ]:
# ---------------- Optional: Google Drive copy with rclone (Kaggle) ----------------
SYNC = False
if ON_KAGGLE and DRIVE_DIR:
    conf = get_secret("RCLONE_CONF")
    if conf:
        if not shutil.which("rclone"):
            import urllib.request, zipfile, io
            ver = "v1.68.2"
            data = urllib.request.urlopen(f"https://downloads.rclone.org/{ver}/rclone-{ver}-linux-amd64.zip").read()
            with zipfile.ZipFile(io.BytesIO(data)) as z:
                with z.open(f"rclone-{ver}-linux-amd64/rclone") as src, open("/usr/local/bin/rclone", "wb") as dst:
                    dst.write(src.read())
            os.chmod("/usr/local/bin/rclone", 0o755)
        os.makedirs(os.path.expanduser("~/.config/rclone"), exist_ok=True)
        with open(os.path.expanduser("~/.config/rclone/rclone.conf"), "w") as f:
            f.write(conf)
        SYNC = True
        # Pull earlier results so --resume continues where the last session stopped.
        # --update never replaces a newer file with an older one.
        subprocess.run(["rclone", "copy", DRIVE_DIR, RESULTS_DIR, "--update"], check=True)
        print("Drive sync on. Files now in RESULTS_DIR:", sorted(os.listdir(RESULTS_DIR)))
    else:
        print("No RCLONE_CONF secret: results stay in", RESULTS_DIR,
              "- download them before the session ends, or use 'Save Version' so Kaggle keeps them.")


def push(include):
    """Copy files matching `include` (e.g. 'pilot*') to Drive. Only your own files,
    so you never overwrite a teammate's shard."""
    if SYNC:
        subprocess.run(["rclone", "copy", RESULTS_DIR, DRIVE_DIR, "--update", "--include", include], check=True)

In [ ]:
# ---------------- Data: POPE (pinned) + only the ~500 COCO images it uses (~80 MB) ----------------
!python scripts/download_data.py
assert os.path.exists(f"{REPO}/data/splits.json"), "splits.json missing: it should come with the repo"

In [ ]:
# ---------------- CPU tests (mock model, < 1 min) ----------------
# Includes a check that rebuilding the split here gives the committed splits.json.
!python -m pytest -q

## Pilot (Phases 2 and 3)
For each model: 100 dev questions × 4 pushback types (paraphrase 0) = 500 forward passes.
It first prints 5 examples in full (exact chat-template text, p(yes), p(no), turn-1 and turn-2 answers) for the hand check,
then a summary. Results: `RESULTS_DIR/pilot_dev_<model>.jsonl` and `RESULTS_DIR/pilot.csv`.
The first run of each model also downloads its weights (Qwen2-VL-2B about 4.5 GB, LLaVA-1.5-7B about 14 GB).
If it stops, run the cell again: `--resume` skips finished questions.

In [ ]:
for m in PILOT_MODELS:
    print(f"\n######## pilot: {m} ########", flush=True)
    !python scripts/pilot.py --model-config configs/models/{m}.yaml --resume
    push("pilot*")

import pandas as pd
pd.read_csv(f"{RESULTS_DIR}/pilot.csv")

## Stop here after the pilot
Check the printed examples and `pilot.csv` with the team:
`mean_mass_t2` (does the model still answer yes/no after pushback?), `flip_rate` (target roughly 20–70%), and `sec_per_question`.
Then set `DEV_MODEL` (and `SHARD` per teammate) in the settings cell and run the cells below.

In [ ]:
# ---------------- Dev run: resumable loop ----------------
assert DEV_MODEL, "Set DEV_MODEL in the settings cell after the pilot"
from vlmsyc.models import load_model
from vlmsyc.paths import load_yaml, results_dir
from vlmsyc.pipeline import output_path, parse_shard, prepare_dev_run, run_to_file, select_shard

model_cfg = load_yaml(f"configs/models/{DEV_MODEL}.yaml")
setup = prepare_dev_run("configs/experiment.yaml", model_cfg, stage="baseline")
k, n = parse_shard(SHARD)
questions = select_shard(setup["questions"], k, n)
model = load_model(model_cfg)
out = output_path(results_dir(), "baseline", "dev", model.name, k, n)
chunk = setup["exp"]["baseline"]["chunk_questions"]
print(f"shard {k}/{n}: {len(questions)} questions x {len(setup['pushbacks'])} pushbacks -> {out}")

while True:
    stats = run_to_file(
        model, questions, setup["pushbacks"], setup["prompt_cfg"], out,
        partition="dev", resume=True, batch_size=model_cfg["batch_size"], max_new_questions=chunk,
        image_loader=setup["image_loader"], seed=setup["seed"], git_commit=setup["git_commit"],
        meta={**setup["meta"], "shard": SHARD, "model": model.describe()}, log_every=5,
    )
    push(out.name)  # copy this shard to Drive after every chunk
    print(time.strftime("%H:%M:%S"), stats, flush=True)
    if stats["remaining"] == 0:
        print("Shard finished.")
        break

## Merge shards (once every teammate's shard is finished)
Needs all shard files in `RESULTS_DIR` (with Drive sync, re-run the rclone cell first to pull them).
Then copy `RESULTS_DIR` back into `results/` in the repo on a laptop and commit.

In [ ]:
assert DEV_MODEL
!python scripts/merge_shards.py --model {DEV_MODEL} --expect-pushbacks 12